In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# =========================================================
# 1. IMPORTS & REPRODUCIBILITY
# =========================================================
import os
import torch
import random
import numpy as np
import pandas as pd
import shutil
import gc
from pathlib import Path
from tqdm import tqdm
from diffusers import StableDiffusionXLPipeline

GLOBAL_SEED = 1234
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

# =========================================================
# 2. CONFIGURATION
# =========================================================
# Set to 101 to cover seeds 42 through 142
SEED_START = 142
SEED_END = 143 
# SEEDS = range(SEED_START, SEED_END + 1)
SEEDS = [142,742,842]

OUTPUT_DIR = Path("/kaggle/working/enhancing_Batch")
OUTPUT_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# 3. PROMPT & TRACKING WORDS DEFINITION
# =========================================================
prompts = [
            "A ball roughly next to a basket"
    ]
    
    # Note: Ensure get_token_indices extracts the correct nouns for each prompt
    # Here we define the tracking words per prompt index
tracking_words = [
        ["ball", "basket"]
    ]

negative_prompt = (
        "same size object, equal size, merged objects, "
        "single subject, blurry, low quality, cartoon, illustration, "
        "3d render, painting, text, watermark"
    )

    # Generate seeds from 42 to 142 inclusive (101 total)
    # seeds = [142,742,842]

# =========================================================
# 4. LOAD SDXL-TURBO (Memory Optimized)
# =========================================================
print(f"\nLoading SDXL-Turbo on {device}...")

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

pipe.set_progress_bar_config(disable=True)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xFormers enabled ✓")
except Exception:
    print("xFormers not available — continuing with standard attention")

# =========================================================
# 5. GENERATION LOOP
# =========================================================
records = []

for p_idx, prompt_text in enumerate(prompts):
    # Subfolder per prompt for research organization
    prompt_folder_name = f"prompt_{p_idx}_" + prompt_text[:20].replace(" ", "_")
    prompt_path = OUTPUT_DIR / prompt_folder_name
    prompt_path.mkdir(exist_ok=True)
    
    words = tracking_words[p_idx]
    print(f"\n🚀 Processing Prompt {p_idx+1}/{len(prompts)}: {prompt_text}")

    for seed_value in tqdm(SEEDS, desc="Seeds"):
        generator = torch.Generator(device=device).manual_seed(seed_value)

        try:
            # SDXL-TURBO SETTINGS: 
            # High steps (50) and Guidance (7.5) will break Turbo quality.
            # Use 1-4 steps and guidance_scale=0.0 for best results.
            image = pipe(
                prompt=prompt_text,
                negative_prompt=negative_prompt,
                num_inference_steps=4,      
                guidance_scale=0.0,         
                generator=generator
            ).images[0]

            filename = f"seed_{seed_value}_stage1.png"
            save_path = prompt_path / filename
            image.save(save_path)

            records.append({
                "prompt_id": p_idx,
                "prompt": prompt_text,
                "seed": seed_value,
                "path": str(save_path),
                "tracking_words": words
            })

            # Explicit memory cleanup per image
            del image
            gc.collect()
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"\nError at Prompt {p_idx}, Seed {seed_value}: {e}")

# =========================================================
# 6. METADATA & EXPORT
# =========================================================
df = pd.DataFrame(records)
df.to_csv(OUTPUT_DIR / "experiment_metadata.csv", index=False)

print(f"\n✨ Batch complete. {len(df)} images saved.")
print(f"Creating ZIP archive...")

shutil.make_archive("/kaggle/working/attend_Batch_Archive", "zip", OUTPUT_DIR)
print("ZIP READY: /kaggle/working/attend_Batch_Archive.zip ✓")


Loading SDXL-Turbo on cuda...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

xFormers not available — continuing with standard attention

🚀 Processing Prompt 1/1: A ball roughly next to a basket


Seeds:   0%|          | 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
Seeds: 100%|██████████| 3/3 [00:05<00:00,  1.88s/it]


✨ Batch complete. 3 images saved.
Creating ZIP archive...
ZIP READY: /kaggle/working/attend_Batch_Archive.zip ✓
